# 1. Project Overview

**Smart RAG Document Assistant**

This notebook implements the complete Retrieval-Augmented Generation (RAG) pipeline:
1. Extract text from PDF documents.
2. Clean and chunk the text.
3. Generate embeddings.
4. Store in ChromaDB.
5. Retrieve relevant chunks using an Ollama LLM.
6. Evaluate the quality of retrieval and generation.

# 2. Dataset Description

The dataset consists of PDF documents placed in `data/documents/`. If this folder is empty, the pipeline will gracefully handle the empty dataset and skip embedding/generation steps until documents are added.

# 3. Imports

In [1]:
import os
import re
import glob
import json
import yaml
import requests
import uuid

from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb

# Load configuration
CONFIG_PATH = os.path.join("..", "backend", "config.yaml")
if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f).get("rag", {})
else:
    # Fallback defaults if config is missing
    config = {
        "chunk_size": 700,
        "chunk_overlap": 100,
        "embedding_model": "all-MiniLM-L6-v2",
        "ollama_model": "llama3.2:3b"
    }

print("Libraries imported and config loaded successfully.")
print(json.dumps(config, indent=2))


c:\Users\moham\OneDrive\Desktop\ITI-PROJECT\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported and config loaded successfully.
{
  "chunk_size": 700,
  "chunk_overlap": 100,
  "embedding_model": "all-MiniLM-L6-v2",
  "ollama_model": "llama3.2:3b"
}


# 4. Load Documents

Load all PDF files from `data/documents/` and extract text page-by-page, keeping metadata (file name, page number, document id).

In [2]:
DOCUMENTS_DIR = os.path.join("..", "data", "documents")
os.makedirs(DOCUMENTS_DIR, exist_ok=True)

pdf_files = sorted(glob.glob(os.path.join(DOCUMENTS_DIR, "*.pdf")))

raw_pages = []
failed_files = []

for pdf_path in pdf_files:
    file_name = os.path.basename(pdf_path)
    # create a stable document ID based on filename
    doc_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, file_name))
    try:
        reader = PdfReader(pdf_path)
        for page_num, page in enumerate(reader.pages, start=1):
            text = page.extract_text()
            if text and text.strip():
                raw_pages.append({
                    "text": text,
                    "metadata": {
                        "file_name": file_name,
                        "page_number": page_num,
                        "document_id": doc_id
                    }
                })
    except Exception as e:
        failed_files.append((file_name, str(e)))

if not pdf_files:
    print(f"No PDFs found in {os.path.abspath(DOCUMENTS_DIR)}")
else:
    print(f"Extracted {len(raw_pages)} page(s) total from {len(pdf_files)} PDF(s).")


Extracted 3 page(s) total from 3 PDF(s).


# 5. Dataset Inspection

Show the number of documents, pages, total extracted text length, and any failed files.

In [3]:
num_docs = len(pdf_files)
num_pages = len(raw_pages)
total_text_length = sum(len(p['text']) for p in raw_pages)

print("=== Dataset Inspection ===")
print(f"Number of documents : {num_docs}")
print(f"Number of pages     : {num_pages}")
print(f"Extracted length    : {total_text_length} characters")

if failed_files:
    print("\nFailed files:")
    for f, err in failed_files:
        print(f"  - {f}: {err}")
else:
    print("\nNo failed files.")


=== Dataset Inspection ===
Number of documents : 3
Number of pages     : 3
Extracted length    : 7435 characters

No failed files.


# 6. Text Cleaning

Clean the text to:
- Remove unnecessary spaces
- Remove duplicated new lines
- Normalize text (without changing the meaning)

In [4]:
def clean_text(text: str) -> str:
    # Replace duplicate newlines with a single newline
    text = re.sub(r'\n{2,}', '\n', text)
    # Replace multiple whitespace spaces (excluding newlines) with a single space
    text = re.sub(r'[^\S\n]+', ' ', text)
    return text.strip()

for page in raw_pages:
    page["text"] = clean_text(page["text"])

if raw_pages:
    print("Text cleaning complete. Sample from first page:")
    print("=" * 60)
    print(raw_pages[0]["text"][:300] + "...")
else:
    print("No text to clean.")


Text cleaning complete. Sample from first page:
Fundamentals of Cell Biology
What is a Cell?
The cell is the fundamental unit of life. All living organisms are composed of one or more cells. Cells were first
discovered by Robert Hooke in 1665 when he observed thin slices of cork under a microscope. The cell
theory, formulated by Schleiden and Sch...


# 7. Chunking

We split the text into manageable chunks.
- **chunk_size (700)**: Large enough to hold a complete paragraph or concept, but small enough to maintain precise relevance.
- **chunk_overlap (100)**: Ensures that context isn't lost across chunk boundaries (e.g., if a sentence gets split).

In [5]:
CHUNK_SIZE = config.get("chunk_size", 700)
CHUNK_OVERLAP = config.get("chunk_overlap", 100)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = []

for page in raw_pages:
    splits = text_splitter.split_text(page["text"])
    for i, chunk_text in enumerate(splits):
        chunks.append({
            "text": chunk_text,
            "metadata": {
                "source": page["metadata"]["file_name"],
                "page": page["metadata"]["page_number"],
                "document_id": page["metadata"]["document_id"],
                "chunk_id": f"{page['metadata']['document_id']}_p{page['metadata']['page_number']}_c{i}"
            }
        })

print(f"Created {len(chunks)} chunks from {len(raw_pages)} pages.")
if chunks:
    print("\nSample chunk metadata:")
    print(chunks[0]["metadata"])


Created 13 chunks from 3 pages.

Sample chunk metadata:
{'source': 'cell_biology.pdf', 'page': 1, 'document_id': '2b9a4200-2897-5e0a-8522-3c607d3a1908', 'chunk_id': '2b9a4200-2897-5e0a-8522-3c607d3a1908_p1_c0'}


# 8. Embeddings

Using `sentence-transformers` to embed chunks into vectors. We use a lightweight model suitable for CPU/GPU execution without huge memory requirements.

In [6]:
MODEL_NAME = config.get("embedding_model", "all-MiniLM-L6-v2")
print(f"Loading embedding model: {MODEL_NAME}")
embedding_model = SentenceTransformer(MODEL_NAME)

embeddings = []
if chunks:
    chunk_texts = [c["text"] for c in chunks]
    print(f"Generating embeddings for {len(chunk_texts)} chunks...")
    embeddings_array = embedding_model.encode(chunk_texts, show_progress_bar=True, batch_size=32)
    embeddings = embeddings_array.tolist()
    print("Embeddings generated.")
else:
    print("No chunks to embed.")


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5980.09it/s]


Generating embeddings for 13 chunks...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.44it/s]

Embeddings generated.


# 9. Chroma Vector Store

Initialize the Chroma vector database and store the chunks, embeddings, and metadata. We persist it to `backend/data/vector_store/` so the backend can reuse it without re-embedding.

In [7]:
PERSIST_DIR = os.path.join("..", "backend", "data", "vector_store")
os.makedirs(PERSIST_DIR, exist_ok=True)

chroma_client = chromadb.PersistentClient(path=PERSIST_DIR)
COLLECTION_NAME = "science_docs"

# We reset the collection to keep this notebook reproducible
try:
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

if chunks and embeddings:
    ids = [c["metadata"]["chunk_id"] for c in chunks]
    documents = [c["text"] for c in chunks]
    metadatas = [c["metadata"] for c in chunks]
    
    collection.add(
        ids=ids,
        documents=documents,
        metadatas=metadatas,
        embeddings=embeddings
    )
    print(f"Stored {collection.count()} vectors into Chroma collection '{COLLECTION_NAME}'.")
else:
    print("No data to store in Chroma.")
    
print(f"Vector database path: {os.path.abspath(PERSIST_DIR)}")


Stored 13 vectors into Chroma collection 'science_docs'.
Vector database path: c:\Users\moham\OneDrive\Desktop\ITI-PROJECT\backend\data\vector_store


# 10. Retrieval Testing

Create a `retrieve` function to embed the question, search ChromaDB, and return the most relevant chunks and sources.

In [8]:
def retrieve(question: str, top_k: int = 4) -> dict:
    if collection.count() == 0:
        return {"documents": [], "metadatas": [], "distances": []}
        
    query_embedding = embedding_model.encode([question]).tolist()
    
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=min(top_k, collection.count()),
    )
    
    return {
        "documents": results["documents"][0],
        "metadatas": results["metadatas"][0],
        "distances": results["distances"][0],
    }

print("Retrieval function defined.")

# Test retrieval if there is data
if collection.count() > 0:
    res = retrieve("What are Newton's laws?", top_k=2)
    print(f"Found {len(res['documents'])} chunks for test query.")
    for m in res['metadatas']:
        print(f" - Source: {m['source']} (Page {m['page']})")


Retrieval function defined.
Found 2 chunks for test query.
 - Source: newtons_laws.pdf (Page 1)
 - Source: newtons_laws.pdf (Page 1)


# 11. Ollama Generation

Connect to the local Ollama LLM and define the RAG prompt to ensure strict context adherence.

In [9]:
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL = config.get("ollama_model", "llama3.2:3b")

RAG_PROMPT_TEMPLATE = """You are a helpful science document assistant. Answer the user's question using ONLY the context provided below.

Rules:
- Answer only from provided context
- Do not use outside knowledge
- If information is missing say:
  "Information not found in the provided documents."

Context:
{context}

Question: {question}

Answer:"""

def call_ollama(prompt: str) -> str:
    url = f"{OLLAMA_BASE_URL}/api/generate"
    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.0}
    }
    try:
        response = requests.post(url, json=payload, timeout=120)
        response.raise_for_status()
        return response.json()["response"].strip()
    except Exception as e:
        return f"Error connecting to Ollama: {e}"

def rag_query(question: str, top_k: int = 4) -> dict:
    retrieval = retrieve(question, top_k=top_k)
    
    docs = retrieval.get("documents", [])
    metas = retrieval.get("metadatas", [])
    
    if not docs:
        context = "No documents found."
        sources = []
    else:
        context_parts = []
        sources = []
        for i, (doc, meta) in enumerate(zip(docs, metas)):
            context_parts.append(f"[Source {i+1} from {meta['source']}, page {meta['page']}]\n{doc}")
            sources.append(f"{meta['source']} (Page {meta['page']})")
        context = "\n\n".join(context_parts)
    
    prompt = RAG_PROMPT_TEMPLATE.format(context=context, question=question)
    answer = call_ollama(prompt)
    
    return {
        "question": question,
        "answer": answer,
        "sources": sources
    }

print("Ollama RAG integration ready.")


Ollama RAG integration ready.


# 12. Evaluation

Test the full pipeline with 10 questions and record the results, including failure analysis.

In [10]:
test_questions = [
    "What is a prokaryotic cell?",
    "What are the four inner planets of our solar system?",
    "What is Newton's First Law?",
    "What is the function of ribosomes?",
    "What is the Great Red Spot on Jupiter?",
    "How does the asteroid belt differ from the Kuiper Belt?",
    "Explain the concept of inertia.",
    "What is the chemical equation for cellular respiration?",
    "What is the capital of Japan?",  # Out of scope - should trigger "Information not found"
    "Who won the World Cup in 2022?" # Out of scope - should trigger "Information not found"
]

evaluation_results = []

print("Running Evaluation...\n")

for q in test_questions:
    res = rag_query(q)
    evaluation_results.append(res)
    print(f"Q: {q}")
    print(f"A: {res['answer']}")
    print(f"Sources: {', '.join(res['sources']) if res['sources'] else 'None'}")
    print("-" * 50)


Running Evaluation...

Q: What is a prokaryotic cell?
A: According to Source 2 from cell_biology.pdf, page 1, a prokaryotic cell is a type of cell that lacks a membrane-bound nucleus; their DNA floats freely in the cytoplasm in a region called the nucleoid.
Sources: cell_biology.pdf (Page 1), cell_biology.pdf (Page 1), cell_biology.pdf (Page 1), cell_biology.pdf (Page 1)
--------------------------------------------------
Q: What are the four inner planets of our solar system?
A: The four inner planets of our solar system are Mercury, Venus, Earth, and Mars.
Sources: solar_system.pdf (Page 1), solar_system.pdf (Page 1), solar_system.pdf (Page 1), solar_system.pdf (Page 1)
--------------------------------------------------
Q: What is Newton's First Law?
A: Newton's First Law states that an object at rest will remain at rest, and an object in motion will continue in motion with constant velocity, unless acted upon by a net external force.
Sources: newtons_laws.pdf (Page 1), newtons_laws.p

## Failure Analysis

**(Manual Notes based on output above)**
- **Retrieval problems:** If chunks lack sufficient context, retrieval quality drops. Overlap helps prevent this.
- **Hallucination cases:** By strictly enforcing the "Information not found" rule in the RAG prompt (and with `temperature=0.0`), hallucinations on out-of-scope questions (like capitals or sports) are effectively mitigated.
- **Improvements made:** Added chunk overlap to preserve sentence boundaries. Added strict prompt guards against hallucination.

# 13. Export Vector Store

Verify the vector database is fully persisted and ready for the backend application.

In [11]:
import glob

print("Checking Vector Store Files:")
for file_path in glob.glob(os.path.join(PERSIST_DIR, "**", "*"), recursive=True):
    if os.path.isfile(file_path):
        size = os.path.getsize(file_path)
        print(f" - {os.path.relpath(file_path, PERSIST_DIR)} ({size} bytes)")

print("\nRAG Pipeline generation complete!")


Checking Vector Store Files:
 - chroma.sqlite3 (339968 bytes)
 - 14a3f016-58a4-47e7-9c7f-eb7428cf2ec1\data_level0.bin (167600 bytes)
 - 14a3f016-58a4-47e7-9c7f-eb7428cf2ec1\header.bin (100 bytes)
 - 14a3f016-58a4-47e7-9c7f-eb7428cf2ec1\length.bin (400 bytes)
 - 14a3f016-58a4-47e7-9c7f-eb7428cf2ec1\link_lists.bin (0 bytes)
 - 245babb6-1e51-49c2-a147-40ee8644e871\data_level0.bin (167600 bytes)
 - 245babb6-1e51-49c2-a147-40ee8644e871\header.bin (100 bytes)
 - 245babb6-1e51-49c2-a147-40ee8644e871\length.bin (400 bytes)
 - 245babb6-1e51-49c2-a147-40ee8644e871\link_lists.bin (0 bytes)
 - dfe1d5d9-d3d4-4c51-a7eb-af4b9947af04\data_level0.bin (167600 bytes)
 - dfe1d5d9-d3d4-4c51-a7eb-af4b9947af04\header.bin (100 bytes)
 - dfe1d5d9-d3d4-4c51-a7eb-af4b9947af04\length.bin (400 bytes)
 - dfe1d5d9-d3d4-4c51-a7eb-af4b9947af04\link_lists.bin (0 bytes)

RAG Pipeline generation complete!
